In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

import statsmodels.formula.api as smf
from sklearn.ensemble import RandomForestClassifier
import numpy as np
from sklearn.linear_model import LinearRegression, LogisticRegression

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, matthews_corrcoef
from kneed import KneeLocator

from matplotlib.lines import Line2D
from matplotlib.patches import Patch

In [2]:
file_path = '/home/imokhtatif/.vscode-server/Chlamy_Project_v2-main/Data/2025_5_phase2.csv'  
df = pd.read_csv(file_path, low_memory=False)

In [3]:
phase2_plates = ['30v1','30v2','30v3','31v1','31v2','31v3','32v1','32v2','32v3','33v1','33v2','33v3']
phase2_df= df[df['plate'].isin(phase2_plates)]

In [4]:
# Make sure que tous les mutants sont communs à toutes les plaques sur chaque régime
plates = ['30v1', '30v2', '30v3']
y2_cols = [f'y2_{i}' for i in range(1, 45)]
light_regimes = phase2_df['light_regime'].unique()

all_filtered = []

for regime in light_regimes:
    subset = phase2_df[
        (phase2_df['light_regime'] == regime) &
        (phase2_df['plate'].isin(plates))
    ]

    # Séparer WT / mutants
    wt_subset = subset[subset['mutant_ID'] == 'WT']
    mutants_subset = subset[subset['mutant_ID'] != 'WT']

    # WT : well_ID communs
    wt_wells_by_plate = {
        plate: set(wt_subset[wt_subset['plate'] == plate]['well_id'])
        for plate in plates
    }
    common_wt_wells = set.intersection(*wt_wells_by_plate.values())
    wt_filtered = wt_subset[wt_subset['well_id'].isin(common_wt_wells)].copy()

    # Mutants : mutant_ID communs
    mutants_by_plate = {
        plate: set(mutants_subset[mutants_subset['plate'] == plate]['mutant_ID'])
        for plate in plates
    }
    common_mutants = set.intersection(*mutants_by_plate.values())
    mutants_filtered = mutants_subset[mutants_subset['mutant_ID'].isin(common_mutants)].copy()

    # Recolle pour ce régime
    regime_filtered = pd.concat([wt_filtered, mutants_filtered], ignore_index=True)
    all_filtered.append(regime_filtered)

# DataFrame final filtré pour tout
clean_df = pd.concat(all_filtered, ignore_index=True)


In [ ]:
# drop last data point
y2_cols = [f'y2_{i}' for i in range(1, 91)] 
def drop_last_valid(row):
    valid = row[y2_cols].last_valid_index()
    # if pd.notna(row[valid]):
    if valid is not None and pd.notna(row[valid]):
        row[valid] = np.nan
    return row

phase2_df1 = phase2_df.apply(drop_last_valid, axis=1)

KeyError: "['y2_90'] not in index"

In [ ]:
#phase2_df1.to_csv('phase2_df2.csv',index=False)

## Run quantile normalization

In [5]:
from scipy import interpolate
from scipy.stats import rankdata

def normalize_quantiles(A, ties=True):
    A = np.asarray(A, dtype=np.float64)
    n_rows, n_cols = A.shape
    if n_cols == 1:
        return A.copy()

    i = np.linspace(0, 1, n_rows)
    S = np.full((n_rows, n_cols), np.nan)
    nobs = np.zeros(n_cols, dtype=int)
    sort_idx = []

    for j in range(n_cols):
        col = A[:, j]
        not_nan = ~np.isnan(col)
        x = col[not_nan]
        nobs[j] = len(x)
        sort_order = np.argsort(x)
        sorted_x = x[sort_order]

        if nobs[j] < n_rows:
            f = interpolate.interp1d(np.linspace(0, 1, nobs[j]), sorted_x,
                                     bounds_error=False, fill_value="extrapolate")
            S[:, j] = f(i)
        else:
            S[:, j] = sorted_x

        sort_idx.append(np.argsort(np.argsort(col[not_nan])))

    m = np.nanmean(S, axis=1)
    A_out = np.full_like(A, np.nan)

    for j in range(n_cols):
        col = A[:, j]
        not_nan = ~np.isnan(col)

        if ties:
            r = rankdata(col[not_nan], method='average')
            quant_pos = (r - 1) / (nobs[j] - 1)
            f = interpolate.interp1d(i, m, bounds_error=False, fill_value="extrapolate")
            A_out[not_nan, j] = f(quant_pos)
        else:
            ranks = sort_idx[j]
            A_out[not_nan, j] = m[ranks.astype(int)]

    return A_out

In [6]:

plates = ['30v1', '30v2', '30v3']
df_30v =phase2_df[phase2_df['plate'].isin(plates)]

group_counts = (
    df_30v.groupby(['plate', 'light_regime', 'mutant_ID', 'mutated_genes'])
    .size()
    .reset_index(name='count')
)

# Step 3: For each plate and light_regime, count how many mutants had 1, 2, ... rows
summary = (
    group_counts.groupby(['light_regime','plate', 'count'])
    .size()
    .reset_index(name='n_mutants')
)

# Optional: Sort for easier reading
summary = summary.sort_values(by=['light_regime','plate', 'count'])

# Show result
summary

,light_regime,plate,count,n_mutants
0,10min-10min,30v1,1,362
1,10min-10min,30v1,7,1
2,10min-10min,30v2,1,363
3,10min-10min,30v2,7,1
4,10min-10min,30v3,1,364
5,10min-10min,30v3,7,1
6,1min-1min,30v1,1,364
7,1min-1min,30v1,7,1
8,1min-1min,30v2,1,364
9,1min-1min,30v2,7,1


## 30 plate 20h_ML


In [7]:
def quantile_normalize_light_regime(df, light_regime, plates, y2_cols, tie_handling=True):
    """
    Quantile-normalize all y2_cols across selected plates within a given light regime.
    
    Parameters:
    - df: pandas DataFrame, full dataset
    - light_regime: str, target light regime (e.g. '20h_ML')
    - plates: list of str, target plate names (e.g. ['30v1', '30v2', '30v3'])
    - y2_cols: list of str, column names like ['y2_1', ..., 'y2_44']
    - tie_handling: bool, passed to normalize_quantiles (default=True)
    
    Returns:
    - df_normalized: pandas DataFrame with normalized y2_cols
    """
    # Filter data
    subset_df = df[(df['light_regime'] == light_regime) & (df['plate'].isin(plates))].copy()
    df_normalized = subset_df.copy()

    for timepoint in y2_cols:
        position_values = []
        valid_plate_indices = {}

        for plate in plates:
            plate_df = subset_df[subset_df['plate'] == plate].copy()

            wt_rows = plate_df[plate_df['mutant_ID'] == 'WT'].copy()
            non_wt_rows = plate_df[plate_df['mutant_ID'] != 'WT'].copy()

            wt_rows = wt_rows.sort_values(['mutant_ID', 'mutated_genes', 'well_id'])
            non_wt_rows = non_wt_rows.sort_values(['mutant_ID', 'mutated_genes'])

            sorted_df = pd.concat([wt_rows, non_wt_rows], axis=0)
            values = sorted_df[timepoint].values
            index = sorted_df.index.values

            position_values.append(values)
            valid_plate_indices[plate] = index

        # Validate shape
        lengths = [len(v) for v in position_values]
        if len(set(lengths)) != 1:
            raise ValueError(f"Length mismatch at {timepoint}: {lengths}")

        matrix = np.column_stack(position_values)
        normalized_matrix = normalize_quantiles(matrix, ties=tie_handling)

        # Write back
        for col_idx, plate in enumerate(plates):
            indices = valid_plate_indices[plate]
            df_normalized.loc[indices, timepoint] = normalized_matrix[:, col_idx]

    return df_normalized


In [8]:

# Sélection de ton régime et plaques
plates = ['30v1', '30v2', '30v3']
y2_cols = [f'y2_{i}' for i in range(1, 45)]

# subset = phase2_df[
#     (phase2_df['light_regime'] == '20h_ML') &
#     (phase2_df['plate'].isin(plates))
# ]

# # Sépare WT et mutants
# wt_subset = subset[subset['mutant_ID'] == 'WT']
# mutants_subset = subset[subset['mutant_ID'] != 'WT']
# wt_wells_by_plate = {
#     plate: set(
#         wt_subset[wt_subset['plate'] == plate]['well_id']
#     )
#     for plate in plates
# }
# common_wt_wells = set.intersection(*wt_wells_by_plate.values())
# wt_filtered = wt_subset[wt_subset['well_id'].isin(common_wt_wells)].copy()
# mutants_by_plate = {
#     plate: set(mutants_subset[mutants_subset['plate'] == plate]['mutant_ID'])
#     for plate in plates
# }
# common_mutants = set.intersection(*mutants_by_plate.values())
# mutants_filtered = mutants_subset[mutants_subset['mutant_ID'].isin(common_mutants)].copy()
# filtered_df = pd.concat([mutants_filtered, wt_filtered], ignore_index=True)


# Run normalization
phase2_30_20h_ML_normalized = quantile_normalize_light_regime(
    df=clean_df,
    light_regime='20h_ML',
    plates=plates,
    y2_cols=y2_cols
)

# View a few columns
phase2_30_20h_ML_normalized[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_35,y2_36,y2_37,y2_38,y2_39,y2_40,y2_41,y2_42,y2_43,y2_44
2124,30v1,WT,WT,N22,0.359489,0.370432,0.422068,0.379664,0.422124,0.385522,...,0.380272,0.366899,0.393307,0.361101,0.353461,0.353477,0.325697,0.349895,0.331257,0.395231
2125,30v1,WT,WT,N12,0.327110,0.397177,0.398182,0.393763,0.405004,0.429297,...,0.394459,0.376856,0.385724,0.406228,0.381166,0.372315,0.412573,0.412532,0.372748,0.382167
2126,30v1,WT,WT,C22,0.364029,0.396709,0.394855,0.429009,0.417669,0.455015,...,0.404819,0.411268,0.373497,0.378902,0.366234,0.414310,0.372316,0.383570,0.395394,0.378111
2127,30v1,WT,WT,C12,0.385954,0.435169,0.435569,0.440826,0.436431,0.477237,...,0.415120,0.420895,0.412341,0.413936,0.389423,0.409568,0.400293,0.424036,0.422005,0.416758
2128,30v1,WT,WT,C03,0.437363,0.459466,0.483766,0.495703,0.501762,0.482038,...,0.470606,0.461321,0.462357,0.466442,0.461102,0.459354,0.472738,0.449000,0.470793,0.449061
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3181,30v3,LMJ.RY0402.186689,Cre06.g268850,F05,0.353077,0.353822,0.388568,0.336816,0.335665,0.407183,...,0.314471,0.338779,0.333338,0.330673,0.336110,0.355954,0.348549,0.350744,0.351943,0.332482
3182,30v3,LMJ.RY0402.154664,Cre03.g205650,F03,0.282184,0.343044,0.354882,0.332626,0.320698,0.349522,...,0.269269,0.252703,0.309953,0.318271,0.283583,0.275734,0.308607,0.288643,0.300703,0.305207
3183,30v3,LMJ.RY0402.221932,Cre06.g278177,F02,0.395304,0.408818,0.425521,0.456502,0.462138,0.463589,...,0.452996,0.430557,0.431652,0.457625,0.456962,0.450953,0.412047,0.430449,0.459041,0.447843
3184,30v3,LMJ.RY0402.104837,Cre06.g274400,F01,0.441306,0.496974,0.487363,0.483443,0.469457,0.500245,...,0.464476,0.436215,0.440098,0.478886,0.445844,0.447802,0.475464,0.454872,0.475083,0.451815


In [9]:
plates = ['30v1', '30v2', '30v3']
phase2_30_20h_ML= phase2_df[(phase2_df['light_regime'] == '20h_ML') & (phase2_df['plate'].isin(plates))].copy()
phase2_30_20h_ML[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_35,y2_36,y2_37,y2_38,y2_39,y2_40,y2_41,y2_42,y2_43,y2_44
2657,30v1,LMJ.RY0402.257186,Cre01.g045150,K19,0.281513,0.272037,0.309737,0.298018,0.292486,0.294218,...,0.240971,0.194258,0.209825,0.221413,0.213910,0.216901,0.207790,0.200755,0.194589,0.214870
2658,30v1,LMJ.RY0402.043448,Cre03.g191350,L02,0.391114,0.418770,0.422897,0.404787,0.431170,0.431461,...,0.368001,0.367208,0.386433,0.325511,0.378521,0.383804,0.359947,0.358591,0.375611,0.388324
2659,30v1,LMJ.RY0402.042123,Cre03.g191350,L01,0.315853,0.303894,0.360714,0.325515,0.357723,0.310149,...,0.287030,0.317499,0.300836,0.286398,0.315075,0.314531,0.284306,0.325729,0.320418,0.313579
2660,30v1,LMJ.RY0402.257268,"Cre03.g191250,Cre03.g191350",K24,0.321144,0.196154,0.297892,0.355610,0.366205,0.298441,...,0.286125,0.325849,0.336694,0.272610,0.223680,0.268322,0.257033,0.229324,0.334017,0.210772
2661,30v1,LMJ.RY0402.043444,Cre02.g105350,K23,0.359966,0.443429,0.424629,0.435300,0.447710,0.335121,...,0.318585,0.282615,0.357764,0.298831,0.245446,0.298727,0.325445,0.303701,0.276411,0.310688
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7795,30v3,LMJ.RY0402.186689,Cre06.g268850,F05,0.367493,0.359454,0.403887,0.348467,0.360262,0.422691,...,0.341485,0.366512,0.359868,0.362535,0.367270,0.389069,0.378941,0.386479,0.387248,0.359560
7796,30v3,LMJ.RY0402.154664,Cre03.g205650,F03,0.291618,0.349599,0.367490,0.345225,0.350279,0.362718,...,0.291758,0.276823,0.330123,0.349770,0.312173,0.308797,0.338993,0.324940,0.330984,0.326571
7797,30v3,LMJ.RY0402.221932,Cre06.g278177,F02,0.408240,0.421780,0.442012,0.473464,0.489900,0.484703,...,0.486512,0.461250,0.460940,0.491288,0.493391,0.486399,0.447745,0.467698,0.500000,0.481830
7798,30v3,LMJ.RY0402.104837,Cre06.g274400,F01,0.456414,0.509683,0.507415,0.503681,0.494795,0.524805,...,0.499607,0.465338,0.469659,0.510812,0.478853,0.479436,0.508558,0.493084,0.516507,0.487846


### 30 plate 20h_HL

In [10]:
# Define inputs
plates = ['30v1', '30v2', '30v3']
y2_cols = [f'y2_{i}' for i in range(1, 45)]


# subset = phase2_df[
#     (phase2_df['light_regime'] == '20h_ML') &
#     (phase2_df['plate'].isin(plates))
# ]
# wt_subset = subset[subset['mutant_ID'] == 'WT']
# mutants_subset = subset[subset['mutant_ID'] != 'WT']
# wt_wells_by_plate = {
#     plate: set(
#         wt_subset[wt_subset['plate'] == plate]['well_id']
#     )
#     for plate in plates
# }
# common_wt_wells = set.intersection(*wt_wells_by_plate.values())
# wt_filtered = wt_subset[wt_subset['well_id'].isin(common_wt_wells)].copy()
# mutants_by_plate = {
#     plate: set(mutants_subset[mutants_subset['plate'] == plate]['mutant_ID'])
#     for plate in plates
# }
# common_mutants = set.intersection(*mutants_by_plate.values())
# mutants_filtered = mutants_subset[mutants_subset['mutant_ID'].isin(common_mutants)].copy()
# filtered_df = pd.concat([mutants_filtered, wt_filtered], ignore_index=True)


# Run normalization
phase2_30_20h_HL_normalized = quantile_normalize_light_regime(
    df=clean_df,
    light_regime='20h_ML',
    plates=plates,
    y2_cols=y2_cols
)

# View a few columns
phase2_30_20h_HL_normalized[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_35,y2_36,y2_37,y2_38,y2_39,y2_40,y2_41,y2_42,y2_43,y2_44
2124,30v1,WT,WT,N22,0.359489,0.370432,0.422068,0.379664,0.422124,0.385522,...,0.380272,0.366899,0.393307,0.361101,0.353461,0.353477,0.325697,0.349895,0.331257,0.395231
2125,30v1,WT,WT,N12,0.327110,0.397177,0.398182,0.393763,0.405004,0.429297,...,0.394459,0.376856,0.385724,0.406228,0.381166,0.372315,0.412573,0.412532,0.372748,0.382167
2126,30v1,WT,WT,C22,0.364029,0.396709,0.394855,0.429009,0.417669,0.455015,...,0.404819,0.411268,0.373497,0.378902,0.366234,0.414310,0.372316,0.383570,0.395394,0.378111
2127,30v1,WT,WT,C12,0.385954,0.435169,0.435569,0.440826,0.436431,0.477237,...,0.415120,0.420895,0.412341,0.413936,0.389423,0.409568,0.400293,0.424036,0.422005,0.416758
2128,30v1,WT,WT,C03,0.437363,0.459466,0.483766,0.495703,0.501762,0.482038,...,0.470606,0.461321,0.462357,0.466442,0.461102,0.459354,0.472738,0.449000,0.470793,0.449061
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3181,30v3,LMJ.RY0402.186689,Cre06.g268850,F05,0.353077,0.353822,0.388568,0.336816,0.335665,0.407183,...,0.314471,0.338779,0.333338,0.330673,0.336110,0.355954,0.348549,0.350744,0.351943,0.332482
3182,30v3,LMJ.RY0402.154664,Cre03.g205650,F03,0.282184,0.343044,0.354882,0.332626,0.320698,0.349522,...,0.269269,0.252703,0.309953,0.318271,0.283583,0.275734,0.308607,0.288643,0.300703,0.305207
3183,30v3,LMJ.RY0402.221932,Cre06.g278177,F02,0.395304,0.408818,0.425521,0.456502,0.462138,0.463589,...,0.452996,0.430557,0.431652,0.457625,0.456962,0.450953,0.412047,0.430449,0.459041,0.447843
3184,30v3,LMJ.RY0402.104837,Cre06.g274400,F01,0.441306,0.496974,0.487363,0.483443,0.469457,0.500245,...,0.464476,0.436215,0.440098,0.478886,0.445844,0.447802,0.475464,0.454872,0.475083,0.451815


In [11]:
plates = ['30v1', '30v2', '30v3']
y2_cols = [f'y2_{i}' for i in range(1, 45)]

phase2_df1 = phase2_df.copy()
# Filter the data
phase2_30_20h_HL = phase2_df1[
    (phase2_df1['light_regime'] == '20h_HL') &
    (phase2_df1['plate'].isin(plates))
].copy()

# Copy to write normalized data
phase2_30_20h_HL_normalized = phase2_30_20h_HL.copy()

# Loop over each y2 column (timepoint)
for timepoint in y2_cols:
    position_values = []
    valid_plate_indices = []

    # Loop through (plate, start_date) technical replicates


    for (plate, start_date), group in phase2_30_20h_HL.groupby(['plate', 'start_date']):


        subset = group.copy()

        # Separate WT and non-WT rows
        wt_rows = subset[subset['mutant_ID'] == 'WT'].copy()
        non_wt_rows = subset[subset['mutant_ID'] != 'WT'].copy()

        # Sort for reproducibility
        wt_rows = wt_rows.sort_values(['mutant_ID', 'mutated_genes', 'well_id', 'start_date'])
        non_wt_rows = non_wt_rows.sort_values(['mutant_ID', 'mutated_genes', 'start_date'])


        # Combine sorted rows
        subset_sorted = pd.concat([wt_rows, non_wt_rows], axis=0)

        # Extract values and index
        values = subset_sorted[timepoint].values
        index = subset_sorted.index.values

        position_values.append(values)
        valid_plate_indices.append(index)

    # Skip timepoint if mismatch or empty
    lengths = [len(v) for v in position_values]
    if len(set(lengths)) != 1 or 0 in lengths:
        print(f"⚠️ Skipping {timepoint} due to mismatch or empty data: lengths = {lengths}")
        continue

    # Quantile normalize
    matrix = np.column_stack(position_values)
    normalized_matrix = normalize_quantiles(matrix, ties=True)

    # Write back
    for col_idx, index in enumerate(valid_plate_indices):
        phase2_30_20h_HL_normalized.loc[index, timepoint] = normalized_matrix[:, col_idx]

# Optional preview
phase2_30_20h_HL_normalized[['plate', 'start_date', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]


⚠️ Skipping y2_1 due to mismatch or empty data: lengths = [371, 370, 355]
⚠️ Skipping y2_2 due to mismatch or empty data: lengths = [371, 370, 355]
⚠️ Skipping y2_3 due to mismatch or empty data: lengths = [371, 370, 355]
⚠️ Skipping y2_4 due to mismatch or empty data: lengths = [371, 370, 355]
⚠️ Skipping y2_5 due to mismatch or empty data: lengths = [371, 370, 355]
⚠️ Skipping y2_6 due to mismatch or empty data: lengths = [371, 370, 355]
⚠️ Skipping y2_7 due to mismatch or empty data: lengths = [371, 370, 355]
⚠️ Skipping y2_8 due to mismatch or empty data: lengths = [371, 370, 355]
⚠️ Skipping y2_9 due to mismatch or empty data: lengths = [371, 370, 355]
⚠️ Skipping y2_10 due to mismatch or empty data: lengths = [371, 370, 355]
⚠️ Skipping y2_11 due to mismatch or empty data: lengths = [371, 370, 355]
⚠️ Skipping y2_12 due to mismatch or empty data: lengths = [371, 370, 355]
⚠️ Skipping y2_13 due to mismatch or empty data: lengths = [371, 370, 355]
⚠️ Skipping y2_14 due to mismatch 

,plate,start_date,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,...,y2_35,y2_36,y2_37,y2_38,y2_39,y2_40,y2_41,y2_42,y2_43,y2_44
2286,30v1,2024-06-11,LMJ.RY0402.185266,"Cre06.g278194 & Cre06.g278195,Cre06.g278194",L23,0.261158,0.302674,0.269819,0.311297,0.258781,...,0.207206,0.258652,0.183290,0.180992,0.190456,0.235021,0.223102,0.228050,0.205703,0.189320
2287,30v1,2024-06-11,LMJ.RY0402.043444,Cre02.g105350,K23,0.261976,0.180966,0.269031,0.281834,0.204365,...,0.241445,0.214363,0.172496,0.191358,0.217706,0.161898,0.103172,0.183074,0.233476,0.119132
2288,30v1,2024-06-11,LMJ.RY0402.255196,"Cre02.g095135,Cre01.g036050",K22,0.177306,0.174390,0.173150,0.152882,0.154171,...,0.161207,0.163010,0.131794,0.150808,0.140349,0.094072,0.135874,0.146660,0.160175,0.135961
2289,30v1,2024-06-11,LMJ.RY0402.256887,"Cre02.g087450,Cre02.g087450 & Cre02.g087500",K21,0.145076,0.190201,0.154984,0.166983,0.161989,...,0.109299,0.101800,0.087153,0.130291,0.078995,0.075366,0.059477,0.109641,0.093304,0.105727
2290,30v1,2024-06-11,LMJ.RY0402.041908,Cre02.g076987,K20,0.219823,0.244487,0.258488,0.246160,0.219703,...,0.153538,0.172028,0.170022,0.167610,0.146201,0.157364,0.177412,0.168105,0.171225,0.138057
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7441,30v3,2024-06-26,LMJ.RY0402.122247,Cre02.g112250,K24,0.112995,0.164315,0.123405,0.183272,0.140043,...,0.024008,0.030600,0.049653,0.024176,0.038814,0.047300,0.049369,0.046387,0.083834,0.052071
7442,30v3,2024-06-26,LMJ.RY0402.144694,"Cre09.g395769,Cre06.g255700",K19,0.202640,0.197923,0.154567,0.174336,0.194629,...,0.100246,0.101845,0.104550,0.084211,0.099197,0.092485,0.101426,0.091638,0.107759,0.124451
7443,30v3,2024-06-26,LMJ.RY0402.089290,Cre04.g219700,K18,0.243672,0.239203,0.204276,0.161710,0.217832,...,0.027085,0.080178,0.029655,0.076058,0.107114,0.057685,0.115373,0.078621,0.055369,0.081562
7444,30v3,2024-06-26,LMJ.RY0402.202371,"Cre13.g589450,Cre01.g045450",K17,0.152816,0.208626,0.203461,0.208951,0.174405,...,0.068078,0.045065,0.058406,-0.004260,0.045452,0.043689,0.069680,0.048535,0.055484,0.067771


In [12]:
import numpy as np
import pandas as pd

# Example normalize_quantiles function if not defined
def normalize_quantiles(matrix, ties=True):
    """
    Normalize the columns of a matrix to have the same distribution.
    """
    ranked = np.argsort(np.argsort(matrix, axis=0), axis=0)
    sorted_matrix = np.sort(matrix, axis=0)
    mean_ranks = np.mean(sorted_matrix, axis=1)

    normalized = np.zeros_like(matrix)
    for i in range(matrix.shape[1]):
        normalized[:, i] = mean_ranks[ranked[:, i]]
    return normalized

# Plates and timepoints
plates = ['30v1', '30v2', '30v3']
y2_cols = [f'y2_{i}' for i in range(1, 45)]

# Filter the data
phase2_30_20h_HL = phase2_df1[
    (phase2_df1['light_regime'] == '20h_HL') &
    (phase2_df1['plate'].isin(plates))
].copy()

# Copy to write normalized data
phase2_30_20h_HL_normalized = phase2_30_20h_HL.copy()

# Dictionary to store mutant ID matrices
mutant_id_matrices = {}

# Loop over each y2 column (timepoint)
for timepoint in y2_cols:
    position_values = []
    valid_plate_indices = []
    mutant_ids_per_column = []

    # Loop through (plate, start_date) technical replicates
    for (plate, start_date), group in phase2_30_20h_HL.groupby(['plate', 'start_date']):
        subset = group.copy()

        # Separate WT and non-WT rows
        wt_rows = subset[subset['mutant_ID'] == 'WT'].copy()
        non_wt_rows = subset[subset['mutant_ID'] != 'WT'].copy()

        # Sort for reproducibility
        wt_rows = wt_rows.sort_values(['mutant_ID', 'mutated_genes', 'well_id', 'start_date'])
        non_wt_rows = non_wt_rows.sort_values(['mutant_ID', 'mutated_genes', 'start_date'])

        # Combine sorted rows
        subset_sorted = pd.concat([wt_rows, non_wt_rows], axis=0)

        # Extract values, index, and mutant_IDs
        values = subset_sorted[timepoint].values
        index = subset_sorted.index.values
        mutant_ids = subset_sorted['mutant_ID'].values

        if len(values) == 0:
            continue

        position_values.append(values)
        valid_plate_indices.append(index)
        mutant_ids_per_column.append(mutant_ids)

    # Skip timepoint if mismatch or empty
    lengths = [len(v) for v in position_values]
    if len(set(lengths)) != 1 or 0 in lengths:
        print(f"⚠️ Skipping {timepoint} due to mismatch or empty data: lengths = {lengths}")
        continue

    # Quantile normalize
    matrix = np.column_stack(position_values)
    normalized_matrix = normalize_quantiles(matrix, ties=True)

    # Store mutant_ID matrix
    mutant_id_matrix = np.column_stack(mutant_ids_per_column)
    mutant_id_matrices[timepoint] = pd.DataFrame(mutant_id_matrix)

    # Write normalized values back
    for col_idx, index in enumerate(valid_plate_indices):
        phase2_30_20h_HL_normalized.loc[index, timepoint] = normalized_matrix[:, col_idx]

# Optional: preview
print("Mutant ID matrix for y2_5:")
print(mutant_id_matrices['y2_5'].head())

# And for normalized values
print(phase2_30_20h_HL_normalized[['plate', 'start_date', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols].head())


⚠️ Skipping y2_1 due to mismatch or empty data: lengths = [371, 370, 355]
⚠️ Skipping y2_2 due to mismatch or empty data: lengths = [371, 370, 355]
⚠️ Skipping y2_3 due to mismatch or empty data: lengths = [371, 370, 355]
⚠️ Skipping y2_4 due to mismatch or empty data: lengths = [371, 370, 355]
⚠️ Skipping y2_5 due to mismatch or empty data: lengths = [371, 370, 355]
⚠️ Skipping y2_6 due to mismatch or empty data: lengths = [371, 370, 355]
⚠️ Skipping y2_7 due to mismatch or empty data: lengths = [371, 370, 355]
⚠️ Skipping y2_8 due to mismatch or empty data: lengths = [371, 370, 355]
⚠️ Skipping y2_9 due to mismatch or empty data: lengths = [371, 370, 355]
⚠️ Skipping y2_10 due to mismatch or empty data: lengths = [371, 370, 355]
⚠️ Skipping y2_11 due to mismatch or empty data: lengths = [371, 370, 355]
⚠️ Skipping y2_12 due to mismatch or empty data: lengths = [371, 370, 355]
⚠️ Skipping y2_13 due to mismatch or empty data: lengths = [371, 370, 355]
⚠️ Skipping y2_14 due to mismatch 

KeyError: 'y2_5'

In [13]:
plates = ['30v1', '30v2', '30v3']
phase2_30_20h_HL= phase2_df1[(phase2_df1['light_regime'] == '20h_HL') & (phase2_df1['plate'].isin(plates))].copy()
phase2_30_20h_HL[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_35,y2_36,y2_37,y2_38,y2_39,y2_40,y2_41,y2_42,y2_43,y2_44
2286,30v1,LMJ.RY0402.185266,"Cre06.g278194 & Cre06.g278195,Cre06.g278194",L23,0.261158,0.302674,0.269819,0.311297,0.258781,0.291688,...,0.207206,0.258652,0.183290,0.180992,0.190456,0.235021,0.223102,0.228050,0.205703,0.189320
2287,30v1,LMJ.RY0402.043444,Cre02.g105350,K23,0.261976,0.180966,0.269031,0.281834,0.204365,0.278185,...,0.241445,0.214363,0.172496,0.191358,0.217706,0.161898,0.103172,0.183074,0.233476,0.119132
2288,30v1,LMJ.RY0402.255196,"Cre02.g095135,Cre01.g036050",K22,0.177306,0.174390,0.173150,0.152882,0.154171,0.193706,...,0.161207,0.163010,0.131794,0.150808,0.140349,0.094072,0.135874,0.146660,0.160175,0.135961
2289,30v1,LMJ.RY0402.256887,"Cre02.g087450,Cre02.g087450 & Cre02.g087500",K21,0.145076,0.190201,0.154984,0.166983,0.161989,0.191459,...,0.109299,0.101800,0.087153,0.130291,0.078995,0.075366,0.059477,0.109641,0.093304,0.105727
2290,30v1,LMJ.RY0402.041908,Cre02.g076987,K20,0.219823,0.244487,0.258488,0.246160,0.219703,0.270274,...,0.153538,0.172028,0.170022,0.167610,0.146201,0.157364,0.177412,0.168105,0.171225,0.138057
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7441,30v3,LMJ.RY0402.122247,Cre02.g112250,K24,0.112995,0.164315,0.123405,0.183272,0.140043,0.128516,...,0.024008,0.030600,0.049653,0.024176,0.038814,0.047300,0.049369,0.046387,0.083834,0.052071
7442,30v3,LMJ.RY0402.144694,"Cre09.g395769,Cre06.g255700",K19,0.202640,0.197923,0.154567,0.174336,0.194629,0.197284,...,0.100246,0.101845,0.104550,0.084211,0.099197,0.092485,0.101426,0.091638,0.107759,0.124451
7443,30v3,LMJ.RY0402.089290,Cre04.g219700,K18,0.243672,0.239203,0.204276,0.161710,0.217832,0.176097,...,0.027085,0.080178,0.029655,0.076058,0.107114,0.057685,0.115373,0.078621,0.055369,0.081562
7444,30v3,LMJ.RY0402.202371,"Cre13.g589450,Cre01.g045450",K17,0.152816,0.208626,0.203461,0.208951,0.174405,0.188963,...,0.068078,0.045065,0.058406,-0.004260,0.045452,0.043689,0.069680,0.048535,0.055484,0.067771


## 30 plate 2h-2h¶

In [15]:
# Define inputs
plates = ['30v1', '30v2', '30v3']
y2_cols = [f'y2_{i}' for i in range(1, 49)]

# Run normalization
phase2_30_2h_2h_normalized = quantile_normalize_light_regime(
    df=clean_df,
    light_regime='2h-2h',
    plates=plates,
    y2_cols=y2_cols
)

# View a few columns
phase2_30_2h_2h_normalized[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols[:10]]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,y2_7,y2_8,y2_9,y2_10
3186,30v1,WT,WT,H12,0.252919,0.241015,0.269849,0.246552,0.266607,0.191307,0.224824,0.247135,0.612566,0.634758
3187,30v1,WT,WT,C03,0.247060,0.320383,0.285715,0.300840,0.310614,0.276359,0.306174,0.306572,0.624586,0.636406
3188,30v1,WT,WT,C12,0.226697,0.213818,0.277141,0.276713,0.244261,0.271469,0.215852,0.203071,0.579228,0.628400
3189,30v1,WT,WT,C22,0.165672,0.213068,0.209031,0.271281,0.239068,0.190602,0.233035,0.205072,0.573268,0.625581
3190,30v1,WT,WT,N12,0.170943,0.175338,0.264773,0.245541,0.278069,0.256442,0.219351,0.223507,0.572054,0.601232
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4249,30v3,LMJ.RY0402.065169,Cre08.g361000,F08,0.213604,0.182621,0.183356,0.193173,0.186076,0.211955,0.179480,0.180340,0.565732,0.592103
4250,30v3,LMJ.RY0402.158356,Cre03.g200500,F09,0.230469,0.228856,0.266696,0.266353,0.233495,0.245916,0.243281,0.260383,0.628285,0.659860
4251,30v3,LMJ.RY0402.213415,Cre01.g030700,F10,0.187756,0.183285,0.169775,0.131758,0.196434,0.174245,0.172330,0.188941,0.535688,0.552347
4252,30v3,LMJ.RY0402.178182,Cre09.g389912,F11,0.152889,0.139361,0.168970,0.183264,0.179529,0.191440,0.176861,0.139226,0.540248,0.589180


In [ ]:
phase2_30_2h_2h= phase2_df1[(phase2_df1['light_regime'] == '2h-2h') & (phase2_df1['plate'].isin(plates))].copy()
phase2_30_2h_2h[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols[:10]]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,y2_7,y2_8,y2_9,y2_10
1149,30v1,LMJ.RY0402.052440,Cre01.g028650,A02,0.149536,0.118418,0.172688,0.119780,0.190143,0.214951,0.130722,0.210320,0.509822,0.557434
1150,30v1,LMJ.RY0402.055420,Cre01.g045150,A03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1151,30v1,LMJ.RY0402.047311,"Cre08.g375800,Cre01.g049900,Cre01.g049900 & Cr...",A04,0.166283,0.186892,0.181170,0.168928,0.106746,0.089804,0.179855,0.119110,0.520025,0.510441
1152,30v1,LMJ.RY0402.054897,Cre02.g095115,A05,0.254300,0.241386,0.253756,0.234268,0.232971,0.250657,0.262042,0.242339,0.614033,0.637819
1153,30v1,LMJ.RY0402.054597,Cre03.g153400,A06,0.221033,0.204123,0.240292,0.164685,0.249126,0.208331,0.258897,0.189821,0.619235,0.662281
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86840,30v2,LMJ.RY0402.228465,Cre09.g398200,P20,0.178483,0.203732,0.244510,0.215436,0.264020,0.171053,0.226065,0.228592,0.540857,0.605745
86841,30v2,LMJ.RY0402.193684,Cre09.g387400,P21,0.137755,0.080398,0.113951,0.130149,0.146465,0.108329,0.151975,0.168080,0.481357,0.528750
86842,30v2,LMJ.RY0402.147464,Cre04.g218500,P22,0.167121,0.228295,0.217918,0.195899,0.222489,0.189124,0.246569,0.262519,0.595913,0.600962
86843,30v2,LMJ.RY0402.232801,Cre08.g385650,P23,0.148319,0.155801,0.169047,0.136979,0.145749,0.143564,0.171515,0.194463,0.542462,0.525771


## plate 30 1min-1min

In [16]:
plates = ['30v1', '30v2', '30v3']
y2_cols = [f'y2_{i}'    for i in range(1, 89)]
phase2_df1 = clean_df.copy()
# Filter the data
phase2_30_1min_1min = phase2_df1[
    (phase2_df1['light_regime'] == '1min-1min') &
    (phase2_df1['plate'].isin(plates))
].copy()

# Copy to write normalized data
phase2_30_1min_1min_normalized = phase2_30_1min_1min.copy()

# Loop over each y2 column (timepoint)
for timepoint in y2_cols:
    position_values = []
    valid_plate_indices = []

    # Loop through (plate, start_date) technical replicates
    for (plate, start_date), group in phase2_30_1min_1min.groupby(['plate', 'start_date']):
        subset = group.copy()

        # Separate WT and non-WT rows
        wt_rows = subset[subset['mutant_ID'] == 'WT'].copy()
        non_wt_rows = subset[subset['mutant_ID'] != 'WT'].copy()

        # Sort for reproducibility
        wt_rows = wt_rows.sort_values(['mutant_ID', 'mutated_genes', 'well_id', 'start_date'])
        non_wt_rows = non_wt_rows.sort_values(['mutant_ID', 'mutated_genes', 'start_date'])

        # Combine sorted rows
        subset_sorted = pd.concat([wt_rows, non_wt_rows], axis=0)

        # Extract values and index
        values = subset_sorted[timepoint].values
        index = subset_sorted.index.values

        position_values.append(values)
        valid_plate_indices.append(index)

    # Skip timepoint if mismatch or empty
    lengths = [len(v) for v in position_values]
    if len(set(lengths)) != 1 or 0 in lengths:
        print(f"⚠️ Skipping {timepoint} due to mismatch or empty data: lengths = {lengths}")
        continue

    # Quantile normalize
    matrix = np.column_stack(position_values)
    normalized_matrix = normalize_quantiles(matrix, ties=True)

    # Write back
    for col_idx, index in enumerate(valid_plate_indices):
        phase2_30_1min_1min_normalized.loc[index, timepoint] = normalized_matrix[:, col_idx]

# Optional preview
phase2_30_1min_1min_normalized[['plate', 'start_date', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,start_date,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,...,y2_79,y2_80,y2_81,y2_82,y2_83,y2_84,y2_85,y2_86,y2_87,y2_88
0,30v1,2024-06-10,WT,WT,H12,0.233359,0.589388,0.192555,0.534576,0.261007,...,0.251670,0.550666,0.213706,0.544156,0.225178,0.538707,0.239775,0.545870,0.241244,0.560130
1,30v1,2024-06-10,WT,WT,C22,0.224359,0.573036,0.234290,0.545424,0.242031,...,0.259716,0.561234,0.231201,0.560013,0.261359,0.563696,0.240885,0.544466,0.241546,0.548129
2,30v1,2024-06-10,WT,WT,C03,0.258719,0.600302,0.266883,0.577004,0.268149,...,0.265402,0.549041,0.269486,0.549784,0.231379,0.566537,0.266483,0.555000,0.232952,0.560311
3,30v1,2024-06-10,WT,WT,C12,0.224188,0.562831,0.227831,0.552000,0.255021,...,0.230434,0.535535,0.239598,0.570527,0.239005,0.557385,0.210914,0.555480,0.262319,0.558549
4,30v1,2024-06-10,WT,WT,N12,0.205883,0.573660,0.213275,0.514932,0.198339,...,0.236317,0.522523,0.173541,0.520197,0.194031,0.537541,0.218342,0.531856,0.196697,0.524486
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1057,30v3,2024-06-25,LMJ.RY0402.144694,"Cre09.g395769,Cre06.g255700",K19,0.204293,0.574029,0.181652,0.521005,0.173068,...,0.196603,0.507818,0.137286,0.534305,0.162992,0.497895,0.191553,0.499352,0.106055,0.506831
1058,30v3,2024-06-25,LMJ.RY0402.201360,Cre01.g049900,K22,0.178024,0.526627,0.192265,0.500608,0.155916,...,0.146137,0.465257,0.178556,0.455316,0.201736,0.494718,0.189247,0.502884,0.180431,0.508395
1059,30v3,2024-06-25,LMJ.RY0402.202371,"Cre13.g589450,Cre01.g045450",K17,0.188686,0.574931,0.195106,0.544514,0.212216,...,0.172823,0.484116,0.180384,0.519258,0.201096,0.508420,0.196609,0.516982,0.130778,0.498441
1060,30v3,2024-06-25,LMJ.RY0402.172592,"Cre11.g476650,Cre09.g388986",K16,0.217462,0.522627,0.185214,0.522167,0.184329,...,0.220234,0.481545,0.192878,0.471929,0.171166,0.510142,0.176326,0.498119,0.197207,0.531944


In [ ]:
y2_cols = [f'y2_{i}' for i in range(1, 92)]
phase2_30_1min_1min= phase2_df1[(phase2_df1['light_regime'] == '1min-1min') & (phase2_df1['plate'].isin(plates))].copy()
phase2_30_1min_1min[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_82,y2_83,y2_84,y2_85,y2_86,y2_87,y2_88,y2_89,y2_90,y2_91
0,30v1,LMJ.RY0402.052440,Cre01.g028650,A02,0.178463,0.550118,0.169154,0.536918,0.157446,0.497102,...,0.514274,0.154958,0.519095,0.165955,0.514616,0.209323,0.503449,NaN,NaN,NaN
1,30v1,LMJ.RY0402.055420,Cre01.g045150,A03,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,30v1,LMJ.RY0402.047311,"Cre08.g375800,Cre01.g049900,Cre01.g049900 & Cr...",A04,0.085688,0.502814,0.120873,0.490352,0.126170,0.462083,...,0.467418,0.073874,0.476327,0.066746,0.476632,0.144827,0.458694,NaN,NaN,NaN
3,30v1,LMJ.RY0402.054897,Cre02.g095115,A05,0.315781,0.649059,0.289019,0.612956,0.306358,0.605436,...,0.608368,0.243948,0.593104,0.247075,0.594532,0.237937,0.580276,NaN,NaN,NaN
4,30v1,LMJ.RY0402.054597,Cre03.g153400,A06,0.296205,0.630581,0.248592,0.589196,0.258415,0.582097,...,0.550243,0.196497,0.543061,0.203080,0.542638,0.212207,0.543212,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94787,30v3,LMJ.RY0402.240458,"Cre13.g588405 & Cre13.g588453,Cre02.g106650",P20,0.213346,0.566172,0.229570,0.516654,0.208745,0.520764,...,0.507074,0.157094,0.539906,0.165638,0.519407,0.162186,0.546528,NaN,NaN,NaN
94788,30v3,LMJ.RY0402.076841,Cre04.g216050,P21,0.302912,0.630209,0.260860,0.603774,0.253227,0.599196,...,0.549287,0.172317,0.526123,0.165680,0.528998,0.132935,0.510404,NaN,NaN,NaN
94789,30v3,LMJ.RY0402.039259,Cre01.g023050,P22,0.163494,0.506909,0.154467,0.505000,0.137106,0.510749,...,0.487017,0.146014,0.480897,0.088540,0.441224,0.102654,0.509137,NaN,NaN,NaN
94790,30v3,LMJ.RY0402.229006,"Cre13.g583250,Cre07.g341900",P23,0.147403,0.578180,0.169607,0.523358,0.079297,0.476844,...,0.511670,0.082184,0.507070,0.125521,0.488418,0.099019,0.507466,NaN,NaN,NaN


In [ ]:
phase2_30_1min_1min[(phase2_30_1min_1min['plate']=='30v1')&(phase2_30_1min_1min['mutant_ID']=='LMJ.RY0402.236577')]

,plate,measurement,start_date,light_regime,dark_threshold,light_threshold,num_frames,i,j,fv_fm,...,measurement_time_173,measurement_time_174,measurement_time_175,measurement_time_176,measurement_time_177,well_id,mutant_ID,feature,mutated_genes,num_mutations
251,30v1,M3,2024-06-10,1min-1min,15.956943,22.289219,180,10,12,0.650724,...,NaN,NaN,NaN,NaN,NaN,K13,LMJ.RY0402.236577,intron,Cre08.g378800,1.0
82117,30v1,M1,2024-06-08,1min-1min,18.779654,25.444115,180,10,12,0.616165,...,NaN,NaN,NaN,NaN,NaN,K13,LMJ.RY0402.236577,intron,Cre08.g378800,1.0


### 30 plate 30s-30s

In [17]:
# Define inputs
plates = ['30v1', '30v2', '30v3']
y2_cols = [f'y2_{i}' for i in range(1, 89)]

# Run normalization
phase2_30_30s_30s_normalized = quantile_normalize_light_regime(
    df=phase2_df1,
    light_regime='30s-30s',
    plates=plates,
    y2_cols=y2_cols
)

# View a few columns
phase2_30_30s_30s_normalized[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_79,y2_80,y2_81,y2_82,y2_83,y2_84,y2_85,y2_86,y2_87,y2_88
4254,30v1,WT,WT,H12,0.244104,0.579786,0.292543,0.543423,0.268808,0.539253,...,0.268086,0.541532,0.228817,0.537402,0.241441,0.505889,0.203305,0.535099,0.244251,0.537708
4255,30v1,WT,WT,C22,0.258604,0.527399,0.195632,0.517734,0.237411,0.504048,...,0.244904,0.536800,0.257151,0.529874,0.279894,0.519749,0.269152,0.514216,0.239833,0.513907
4256,30v1,WT,WT,C03,0.295753,0.566938,0.288622,0.573538,0.277477,0.556588,...,0.278929,0.555533,0.263872,0.574014,0.239094,0.543367,0.248318,0.541960,0.247474,0.561750
4257,30v1,WT,WT,C12,0.244469,0.562091,0.290648,0.552065,0.265711,0.542476,...,0.262533,0.512030,0.269469,0.542335,0.242552,0.526145,0.272565,0.529322,0.262359,0.528514
4258,30v1,WT,WT,N12,0.251553,0.528351,0.205747,0.481251,0.235350,0.539129,...,0.229649,0.492265,0.255809,0.532197,0.222034,0.472835,0.157579,0.495811,0.248868,0.484684
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5317,30v3,LMJ.RY0402.186689,Cre06.g268850,F05,0.215989,0.562091,0.226521,0.503869,0.209897,0.510018,...,0.210948,0.483438,0.199091,0.494069,0.219472,0.477735,0.189138,0.486893,0.206426,0.467689
5318,30v3,LMJ.RY0402.154664,Cre03.g205650,F03,0.169388,0.514688,0.187585,0.496538,0.211118,0.492439,...,0.141499,0.471071,0.181629,0.462261,0.210216,0.468196,0.181982,0.472843,0.156416,0.450007
5319,30v3,LMJ.RY0402.221932,Cre06.g278177,F02,0.317408,0.587465,0.305291,0.564663,0.301149,0.551936,...,0.291887,0.587933,0.308827,0.553474,0.297474,0.575596,0.297514,0.567506,0.299573,0.554896
5320,30v3,LMJ.RY0402.178182,Cre09.g389912,F11,0.182908,0.490153,0.192414,0.479913,0.197087,0.494128,...,0.162666,0.485508,0.177524,0.459834,0.146168,0.460717,0.149586,0.452145,0.155458,0.426219


In [ ]:
y2_cols = [f'y2_{i}' for i in range(1, 89)]
phase2_30_30s_30s= phase2_df1[(phase2_df1['light_regime'] == '30s-30s') & (phase2_df1['plate'].isin(plates))].copy()
phase2_30_30s_30s[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_79,y2_80,y2_81,y2_82,y2_83,y2_84,y2_85,y2_86,y2_87,y2_88
17618,30v3,LMJ.RY0402.047723,"Cre03.g183150 & Cre03.g183200,Cre03.g183150,Cr...",A02,0.420936,0.655267,0.390790,0.621872,0.378465,0.639171,...,0.346719,0.606152,0.338125,0.631475,0.327414,0.612326,0.335618,0.606576,0.331829,0.604739
17619,30v3,LMJ.RY0402.064780,Cre03.g192450,A03,0.165749,0.479098,0.121231,0.468119,0.195359,0.445142,...,0.178492,0.450141,0.121185,0.450676,0.150316,0.474586,0.130775,0.406174,0.108494,0.438719
17620,30v3,LMJ.RY0402.193315,Cre01.g025400,A04,0.213003,0.509001,0.249679,0.504075,0.207678,0.473582,...,0.202284,0.468827,0.158824,0.478704,0.214800,0.483906,0.230056,0.461946,0.188345,0.503438
17621,30v3,LMJ.RY0402.110865,Cre01.g039550,A05,0.192051,0.497267,0.095375,0.445238,0.168908,0.408339,...,0.080697,0.365304,0.093557,0.444062,0.117235,0.388450,0.195765,0.302987,0.089800,0.312022
17622,30v3,LMJ.RY0402.074645,Cre01.g001800,A06,0.176638,0.485322,0.213517,0.474869,0.205212,0.500576,...,0.154120,0.440459,0.157786,0.465356,0.158112,0.437007,0.148491,0.444521,0.141617,0.433360
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108527,30v2,LMJ.RY0402.228465,Cre09.g398200,P20,0.230111,0.542900,0.210873,0.527458,0.210943,0.500793,...,0.217784,0.510020,0.222708,0.482240,0.231461,0.511648,0.177852,0.517925,0.201273,0.513579
108528,30v2,LMJ.RY0402.193684,Cre09.g387400,P21,0.171363,0.480203,0.098401,0.461616,0.142675,0.442858,...,0.125780,0.485039,0.119794,0.423354,0.157176,0.418609,0.079760,0.459043,0.090184,0.474501
108529,30v2,LMJ.RY0402.147464,Cre04.g218500,P22,0.227096,0.523295,0.217629,0.533985,0.231998,0.502400,...,0.186435,0.528623,0.190963,0.528804,0.192836,0.477209,0.219786,0.477641,0.210296,0.498598
108530,30v2,LMJ.RY0402.232801,Cre08.g385650,P23,0.107310,0.458658,0.140347,0.422454,0.139969,0.461889,...,0.132857,0.461449,0.144200,0.449290,0.129569,0.430041,0.096104,0.454487,0.137321,0.413907


### plate 5min-5min

In [24]:
plates = ['30v1', '30v2', '30v3']
y2_cols = [f'y2_{i}' for i in range(1, 89)]
phase1_df1 = clean_df.copy()   
# Filter the data
phase2_30_5min_5min = phase2_df1[
    (phase2_df1['light_regime'] == '5min-5min') &
    (phase2_df1['plate'].isin(plates))
].copy()

# Copy to write normalized data
phase2_30_5min_5min_normalized = phase2_30_5min_5min.copy()

# Loop over each y2 column (timepoint)
for timepoint in y2_cols:
    position_values = []
    valid_plate_indices = []

    # Loop through (plate, start_date) technical replicates
    for (plate, start_date), group in phase2_30_5min_5min.groupby(['plate', 'start_date']):
        subset = group.copy()

        # Separate WT and non-WT rows
        wt_rows = subset[subset['mutant_ID'] == 'WT'].copy()
        non_wt_rows = subset[subset['mutant_ID'] != 'WT'].copy()

        # Sort for reproducibility
        wt_rows = wt_rows.sort_values(['mutant_ID', 'mutated_genes', 'well_id', 'start_date'])
        non_wt_rows = non_wt_rows.sort_values(['mutant_ID', 'mutated_genes', 'start_date'])

        # Combine sorted rows
        subset_sorted = pd.concat([wt_rows, non_wt_rows], axis=0)

        # Extract values and index
        values = subset_sorted[timepoint].values
        index = subset_sorted.index.values

        position_values.append(values)
        valid_plate_indices.append(index)

    # Skip timepoint if mismatch or empty
    lengths = [len(v) for v in position_values]
    if len(set(lengths)) != 1 or 0 in lengths:
        print(f"⚠️ Skipping {timepoint} due to mismatch or empty data: lengths = {lengths}")
        continue

    # Quantile normalize
    matrix = np.column_stack(position_values)
    normalized_matrix = normalize_quantiles(matrix, ties=True)

    # Write back
    for col_idx, index in enumerate(valid_plate_indices):
        phase2_30_5min_5min_normalized.loc[index, timepoint] = normalized_matrix[:, col_idx]

# Optional preview
phase2_30_5min_5min_normalized[['plate', 'start_date', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,start_date,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,...,y2_79,y2_80,y2_81,y2_82,y2_83,y2_84,y2_85,y2_86,y2_87,y2_88
6366,30v2,2024-06-24,LMJ.RY0402.245354,Cre03.g149250,A08,0.167979,0.586999,0.143754,0.585978,0.150606,...,0.116271,0.532404,0.110856,0.525584,0.130930,0.521663,0.122436,0.535597,0.099205,0.534133
6367,30v2,2024-06-24,LMJ.RY0402.137912,"Cre03.g176961,Cre14.g613652",A09,0.314577,0.664724,0.261087,0.626303,0.260566,...,0.200090,0.620825,0.202991,0.606722,0.193401,0.602296,0.181671,0.607867,0.177142,0.615632
6368,30v2,2024-06-24,LMJ.RY0402.189784,Cre01.g001800,A03,0.203215,0.631644,0.208497,0.617467,0.198927,...,0.171346,0.589799,0.148598,0.581646,0.183454,0.582073,0.167334,0.591783,0.147007,0.585729
6369,30v2,2024-06-24,LMJ.RY0402.228483,Cre06.g285900,F10,0.192899,0.591772,0.195962,0.573239,0.182418,...,0.155078,0.557238,0.143426,0.557500,0.135494,0.551911,0.126683,0.551119,0.144293,0.538408
6370,30v2,2024-06-24,LMJ.RY0402.151780,Cre09.g389912,F11,0.227672,0.645133,0.231455,0.622343,0.204701,...,0.158083,0.579757,0.153892,0.578288,0.134348,0.571906,0.149985,0.582819,0.165996,0.579228
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50916,30v3,2024-11-26,LMJ.RY0402.113112,Cre02.g119050,K20,0.263988,0.602281,0.250997,0.597465,0.204431,...,0.109859,0.583497,0.092505,0.585442,0.091137,0.597976,0.098280,0.562347,0.079544,0.546057
50917,30v3,2024-11-26,LMJ.RY0402.256955,Cre07.g341900,K21,0.208804,0.640162,0.183000,0.578053,0.184469,...,0.132958,0.540263,0.129174,0.555700,0.074665,0.542915,0.060975,0.541583,0.152505,0.548101
50918,30v3,2024-11-26,LMJ.RY0402.077556,"Cre02.g075900,Cre09.g391753",K12,0.258257,0.595524,0.219103,0.573155,0.216953,...,0.174791,0.545451,0.156594,0.516511,0.176051,0.539788,0.198329,0.547961,0.158742,0.517527
50919,30v3,2024-11-26,LMJ.RY0402.069599,Cre09.g391753,P24,0.243978,0.617871,0.208328,0.563438,0.239037,...,0.154867,0.532967,0.128067,0.574720,0.148584,0.538920,0.153872,0.553972,0.168523,0.550748


In [19]:
plates = ['30v1', '30v2','30v3']
y2_cols = [f'y2_{i}' for i in range(1, 90)]
phase2_30_5min_5min= phase2_df1[(phase2_df1['light_regime'] == '5min-5min') & (phase2_df1['plate'].isin(plates))].copy()
phase2_30_5min_5min[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_80,y2_81,y2_82,y2_83,y2_84,y2_85,y2_86,y2_87,y2_88,y2_89


## 30 plate 1min-5min

In [22]:
plates = ['30v1', '30v2', '30v3']
y2_cols = [f'y2_{i}' for i in range(1, 89)]
phase2_df1 = phase2_df.copy()
# Filter the data
phase2_30_1min_5min = phase2_df1[
    (phase2_df1['light_regime'] == '1min-5min') &
    (phase2_df1['plate'].isin(plates))
].copy()

# Copy to write normalized data
phase2_30_1min_5min_normalized = phase2_30_1min_5min.copy()

# Loop over each y2 column (timepoint)
for timepoint in y2_cols:
    position_values = []
    valid_plate_indices = []

    # Loop through (plate, start_date) technical replicates
    for (plate, start_date), group in phase2_30_1min_5min.groupby(['plate', 'start_date']):
        subset = group.copy()

        # Separate WT and non-WT rows
        wt_rows = subset[subset['mutant_ID'] == 'WT'].copy()
        non_wt_rows = subset[subset['mutant_ID'] != 'WT'].copy()

        # Sort for reproducibility
        wt_rows = wt_rows.sort_values(['mutant_ID', 'mutated_genes', 'well_id', 'start_date'])
        non_wt_rows = non_wt_rows.sort_values(['mutant_ID', 'mutated_genes', 'start_date'])

        # Combine sorted rows
        subset_sorted = pd.concat([wt_rows, non_wt_rows], axis=0)

        # Extract values and index
        values = subset_sorted[timepoint].values
        index = subset_sorted.index.values

        position_values.append(values)
        valid_plate_indices.append(index)

    # Skip timepoint if mismatch or empty
    lengths = [len(v) for v in position_values]
    if len(set(lengths)) != 1 or 0 in lengths:
        print(f"⚠️ Skipping {timepoint} due to mismatch or empty data: lengths = {lengths}")
        continue

    # Quantile normalize
    matrix = np.column_stack(position_values)
    normalized_matrix = normalize_quantiles(matrix, ties=True)

    # Write back
    for col_idx, index in enumerate(valid_plate_indices):
        phase2_30_1min_5min_normalized.loc[index, timepoint] = normalized_matrix[:, col_idx]

# Optional preview
phase2_30_1min_5min_normalized[['plate', 'start_date', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,start_date,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,...,y2_79,y2_80,y2_81,y2_82,y2_83,y2_84,y2_85,y2_86,y2_87,y2_88
5995,30v2,2024-06-23,LMJ.RY0402.189784,Cre01.g001800,A03,0.236734,0.659412,0.192115,0.654803,0.194157,...,0.131038,0.637762,0.159600,0.627948,0.133328,0.625667,0.150414,0.618954,0.131703,0.643828
5996,30v2,2024-06-23,LMJ.RY0402.164384,Cre09.g392500,A05,0.227953,0.633813,0.182391,0.628088,0.180167,...,0.136429,0.630088,0.148251,0.623121,0.132547,0.629940,0.126749,0.643192,0.108109,0.627399
5997,30v2,2024-06-23,LMJ.RY0402.256921,"Cre05.g245700,Cre08.g361063,Cre08.g361000",P24,0.151252,0.565835,0.159670,0.585059,0.102921,...,0.017700,0.550254,0.079966,0.590698,0.080623,0.587753,0.044713,0.584105,0.077036,0.556150
5998,30v2,2024-06-23,LMJ.RY0402.201666,Cre06.g278194,K22,0.094862,0.559294,0.104101,0.545612,0.092498,...,0.035102,0.530095,0.052555,0.569508,0.051183,0.537876,0.081634,0.555019,0.076711,0.552587
5999,30v2,2024-06-23,LMJ.RY0402.207167,Cre03.g185500,K21,0.223368,0.633435,0.181139,0.597499,0.173730,...,0.118539,0.604727,0.115234,0.604212,0.113468,0.593953,0.125399,0.587355,0.134657,0.601526
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50545,30v3,2024-11-25,LMJ.RY0402.221932,Cre06.g278177,F02,0.204673,0.651537,0.176109,0.651197,0.163849,...,0.107593,0.661894,0.161522,0.648908,0.125106,0.634839,0.117814,0.618954,0.107465,0.628138
50546,30v3,2024-11-25,LMJ.RY0402.104837,Cre06.g274400,F01,0.137073,0.599057,0.160574,0.584763,0.134941,...,0.054692,0.646702,0.119139,0.618853,0.122259,0.621561,0.052399,0.627284,0.058181,0.666504
50547,30v3,2024-11-25,LMJ.RY0402.213415,Cre01.g030700,F10,0.123194,0.596137,0.123751,0.581474,0.130787,...,0.128360,0.594703,0.117444,0.585885,0.117538,0.591618,0.108136,0.584267,0.083038,0.581318
50548,30v3,2024-11-25,LMJ.RY0402.039259,Cre01.g023050,P22,0.337770,0.669860,0.183833,0.633647,0.307680,...,0.111188,0.604162,0.158850,0.585207,0.230147,0.595796,0.241987,0.571690,0.155158,0.617288


In [23]:
plates = ['30v1', '30v2','30v3']
y2_cols = [f'y2_{i}' for i in range(1, 90)]
phase2_30_1min_5min= phase2_df1[(phase2_df1['light_regime'] == '1min-5min') & (phase2_df1['plate'].isin(plates))].copy()
phase2_30_1min_5min[['plate', 'mutant_ID', 'mutated_genes', 'well_id'] + y2_cols]

,plate,mutant_ID,mutated_genes,well_id,y2_1,y2_2,y2_3,y2_4,y2_5,y2_6,...,y2_80,y2_81,y2_82,y2_83,y2_84,y2_85,y2_86,y2_87,y2_88,y2_89
5995,30v2,LMJ.RY0402.189784,Cre01.g001800,A03,0.286091,0.662745,0.224805,0.656407,0.225504,0.657713,...,0.628206,0.214087,0.616775,0.178378,0.614558,0.201466,0.604706,0.176365,0.638405,NaN
5996,30v2,LMJ.RY0402.164384,Cre09.g392500,A05,0.274217,0.624538,0.210922,0.616918,0.206208,0.594279,...,0.617104,0.197029,0.610256,0.177327,0.619713,0.168734,0.640524,0.144387,0.614812,NaN
5997,30v2,LMJ.RY0402.256921,"Cre05.g245700,Cre08.g361063,Cre08.g361000",P24,0.171834,0.543895,0.184157,0.559649,0.107456,0.532339,...,0.512429,0.105626,0.565355,0.110456,0.560360,0.066051,0.556617,0.104654,0.519208,NaN
5998,30v2,LMJ.RY0402.201666,Cre06.g278194,K22,0.103579,0.535642,0.116544,0.513996,0.093680,0.495312,...,0.484514,0.072762,0.538640,0.075930,0.498504,0.114176,0.521717,0.104518,0.517017,NaN
5999,30v2,LMJ.RY0402.207167,Cre03.g185500,K21,0.267307,0.624253,0.209545,0.575969,0.199411,0.598911,...,0.578563,0.152050,0.583007,0.150445,0.569674,0.167419,0.560147,0.180291,0.578130,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50545,30v3,LMJ.RY0402.221932,Cre06.g278177,F02,0.166129,0.652894,0.148473,0.651766,0.141500,0.643353,...,0.660272,0.105437,0.647889,0.083520,0.642419,0.078958,0.633202,0.071308,0.641169,NaN
50546,30v3,LMJ.RY0402.104837,Cre06.g274400,F01,0.120902,0.620433,0.136165,0.610285,0.121787,0.633229,...,0.649685,0.080963,0.633379,0.082348,0.634639,0.028518,0.637949,0.035992,0.660517,NaN
50547,30v3,LMJ.RY0402.213415,Cre01.g030700,F10,0.105194,0.618932,0.108863,0.607712,0.118254,0.624437,...,0.624583,0.080627,0.612891,0.079083,0.617018,0.071164,0.611860,0.053374,0.610660,NaN
50548,30v3,LMJ.RY0402.039259,Cre01.g023050,P22,0.244520,0.661683,0.155292,0.642827,0.226093,0.592570,...,0.630576,0.104884,0.612015,0.149946,0.620017,0.172030,0.601625,0.100780,0.633501,NaN


In [25]:
phase2_30_quantile1= pd.concat([
    phase2_30_20h_ML_normalized,
    phase2_30_20h_HL_normalized,
    phase2_30_2h_2h_normalized,
    phase2_30_1min_1min_normalized,
    phase2_30_30s_30s_normalized,
    phase2_30_5min_5min_normalized,
    phase2_30_1min_5min_normalized
], ignore_index=True)

In [26]:
phase2_30_quantile1.to_csv('/home/imokhtatif/.vscode-server/Chlamy_Project_v2-main/Quantile_normalization/phase2_30_quantile1.csv',index=False)

In [27]:
phase2_30_quantile1.shape

(6840, 467)

In [28]:
plates = ['30v1', '30v2','30v3']
data=phase2_df1[(phase2_df1['plate'].isin(plates))&(phase2_df1['light_regime']!='10min-10min')]
data.shape

(6969, 467)